# Gao et al. (2026) on dandiset 001683: evoked potentials and PLI in wild-type rats

Paper. Gao H, Frake A, Durand DM, He B (2026). Transcranial focused ultrasound stimulation
targeting white matter inhibits seizures in a rat model of epilepsy. *IEEE Trans Neural Syst Rehabil
Eng* 34, 251-259. [doi:10.1109/tnsre.2025.3644273](https://doi.org/10.1109/tnsre.2025.3644273). The
claims file was written from the NIH author manuscript (PMC12848950) and the supplement.

Dandiset. [DANDI:001683](https://dandiarchive.org/dandiset/001683/0.260109.2104), version
0.260109.2104, the only published version (the paper cites none; see `seizure_metrics.ipynb`).

The wild-type phase, before any 4-AP: the target-confirmation recordings (`EP data`: a resting
`BASELINE` recording and one recording per transducer position, each with about 200 bursts, from
the 64-channel intracranial array in left S1), and one EEG recording per rat labelled
`EEG data for PLV analysis` (32 skull channels, `*_1Hz_20ms_CC`).

| Claim | Tested here |
|---|---|
| fig5b-tfus-evoked-potentials | yes |
| fig5cdj-wild-type-gamma-pli-decrease | blocked (data_missing): no EEG of the resting baseline (the sham); the tFUS part is run |
| fig5ghi-low-band-pli-s1-only | yes |

In [ ]:
import re
import time
import warnings
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
import multiprocessing as mp

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import tfus

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)

started = time.time()
assets = tfus.list_assets()
small = assets[assets["size"] < 2e9]
metas = tfus.read_all_meta(small.asset_id.tolist())
ep = [m for m in metas if m["kind"] == "EP data"]
plv = [m for m in metas if m["kind"].startswith("EEG data for PLV")]
print(f"dandiset {tfus.DANDISET_ID} version {tfus.DANDISET_VERSION}: {len(small)} files under 2 GB; "
      f"EP files {len(ep)}, PLV files {len(plv)}, other {len(metas) - len(ep) - len(plv)}; read in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
def position(m):
    lab = m["label"].split("_", 1)[1]
    if lab == "BASELINE":
        return "BASELINE"
    lab = lab.replace("_", "")
    for key in ("CENTER2", "CENTER", "SHALLOW1mm", "DEEP1mm", "FRONT05mm", "FRONT1mm", "BACK05mm", "BACK1mm", "LSIDE05mm", "RSIDE05mm"):
        if "S1" in lab:
            return "S1:" + lab
        if key in lab:
            return key
    return "?" + lab

E = pd.DataFrame([{"subject": m["subject"], "label": m["label"], "position": position(m), "start": m["start"],
                   "n_trials": len(m["intervals"]), "seconds": m["n_samples"] / m["rate"], "channels": m["n_channels"],
                   "spacing": np.median(np.diff(np.sort(m["intervals"].start))), "window": np.median(m["intervals"].stop - m["intervals"].start),
                   "group": m["group"], "asset_id": m["asset_id"]} for m in ep]).sort_values(["subject", "start"])
print("EP files per rat:", E.groupby("subject").size().to_dict())
print("Rats with EP files:", E.subject.nunique(), "(paper: 23 wild-type rats, Fig. 5B N = 23)")
print("Positions recorded:", E.position.value_counts().to_dict())
print("Channels:", E.channels.unique().tolist(), "; locations:", sorted({tuple(m['electrodes'].location.unique()) for m in ep}),
      "; rate:", sorted({m['rate'] for m in ep}))
print("Interval spacing (s):", E.spacing.round(3).value_counts().to_dict(), "; interval length (s):", E.window.round(3).value_counts().to_dict())
print("Trials per file:", E.groupby("position").n_trials.agg(["min", "median", "max"]).to_string())
print("Tags of the EP intervals:", sorted({t for m in ep for t in m["intervals"].tag}))
P = pd.DataFrame([{"subject": m["subject"], "label": m["label"], "start": m["start"], "n_trials": len(m["intervals"]),
                   "spacing": np.median(np.diff(np.sort(m["intervals"].start))), "seconds": m["n_samples"] / m["rate"],
                   "channels": m["n_channels"]} for m in plv]).sort_values("subject")
print("\nPLV files:", len(P), "rats; labels:", sorted(P.label.str.split("_", n=1).str[1].unique()))
print(P.to_string(index=False))
print("Rats with EP files but no PLV file:", sorted(set(E.subject) - set(P.subject)))
same = all(m["electrodes"][["x", "y"]].equals(plv[0]["electrodes"][["x", "y"]]) for m in plv)
print("PLV electrode tables all the same:", same)
other = [m for m in metas if m["kind"] not in ("EP data",) and not m["kind"].startswith("EEG data for PLV")]
print("Other kinds among the small files:", sorted({m['kind'] for m in other}))

The EP files hold the 64 intracranial channels of left S1 only; there is no EEG in them. The
resting `BASELINE` recordings last about 4 to 5 minutes and carry 120 to 151 intervals every 2 s,
tagged `tFUS` like the stimulated ones. The PLV EEG files are recordings with 20 ms pulses
(`1Hz_20ms_CC`), the pulse of the epilepsy experiment (Methods D), made during the wild-type
phase, of 20 of the 23 rats. Their intervals come every 2 s in most files, every 1 s in a few. No
EEG recording of a resting baseline (the sham of Figs. 5E-F) is in the dandiset.

In [ ]:
# Alignment of the intervals with the response: the mean EEG of the PLV files and the mean LFP of the
# CENTER files around the interval starts, baseline-subtracted, in 10 ms bins.
def mean_response(m, n=100, pre=0.05, post=0.2):
    x = tfus.read_epochs(m, np.sort(m["intervals"].start.to_numpy())[:n], pre=pre, post=post)
    x = x - x[:, :int(pre * tfus.RATE)].mean(axis=1, keepdims=True)
    return np.abs(x.mean(axis=0)).max(axis=1) * 1e6     # peak over channels of the trial mean, uV

rows = []
for kind, files in (("PLV EEG", plv[:5]), ("EP LFP CENTER", [m for m in ep if position(m) == "CENTER"][:5]),
                    ("EP LFP BASELINE", [m for m in ep if position(m) == "BASELINE"][:5])):
    for m in files:
        r = mean_response(m)
        t = (np.arange(len(r)) - int(0.05 * tfus.RATE)) / tfus.RATE * 1000
        rows.append({"kind": kind, "subject": m["subject"], "pre -50-0 ms": r[t < 0].max(), "0-40 ms": r[(t >= 0) & (t < 40)].max(),
                     "40-200 ms": r[t >= 40].max()})
print("Peak over channels of |trial mean| (uV), first 100 intervals of five files of each kind:")
print(pd.DataFrame(rows).round(2).to_string(index=False))

In [ ]:
# Alignment of the EP intervals with the bursts (verifier, round 1). The intervals last 2.2 s and start
# 2 s apart; a burst at another lag would still show at a fixed lag of the 2 s cycle. Two checks on the
# first 40 cycles of every CENTER and BASELINE file: (1) the RMS over the 64 channels of the
# trial-mean LFP (3-300 Hz) over the whole cycle, -100 to 1900 ms from the interval start, in 10 ms
# bins; (2) the power at 1400-1520 Hz, the band of the bursts' 1.5 kHz pulse repetition frequency (just
# under the 1526 Hz Nyquist frequency), where an ultrasound artifact would mark the 67 ms burst.
from scipy import signal as sg
for kind, files in (("EP", ep), ("PLV", plv)):
    L = np.concatenate([(m["intervals"].stop - m["intervals"].start).to_numpy() for m in files])
    print(f"{kind} files, interval stop - start: {L.min():.4f} to {L.max():.4f} s; start spacing:",
          sorted({round(float(np.median(np.diff(np.sort(m['intervals'].start)))), 3) for m in files}))
SOS_LFP = sg.butter(4, [3, 300], btype="bandpass", fs=tfus.RATE, output="sos")
SOS_PRF = sg.butter(4, [1400, 1520], btype="bandpass", fs=tfus.RATE, output="sos")

def cycle_profile(m, n=40):
    on = np.sort(m["intervals"].start.to_numpy())[1:n + 1]
    pre, L, b = int(0.1 * tfus.RATE), int(2.0 * tfus.RATE), int(0.01 * tfus.RATE)
    i0, i1 = int(on[0] * tfus.RATE) - pre, int(on[-1] * tfus.RATE) - pre + L
    x = tfus.read_rows(m, i0, i1, threads=8)
    idx = [int(t * tfus.RATE) - pre - i0 for t in on]
    lfp = sg.sosfiltfilt(SOS_LFP, x, axis=0)
    mu = np.mean([lfp[i:i + L] for i in idx], axis=0)
    rms = np.sqrt((mu ** 2).mean(1)) * 1e6
    hf = (sg.sosfiltfilt(SOS_PRF, x, axis=0) ** 2).mean(1)
    pw = np.mean([hf[i:i + L] for i in idx], axis=0)
    nb = L // b
    return rms[:nb * b].reshape(nb, b).mean(1), pw[:nb * b].reshape(nb, b).mean(1)

t_cyc = np.arange(int(2.0 * tfus.RATE) // int(0.01 * tfus.RATE)) * 10 - 100
prof = {}
with ThreadPoolExecutor(4) as ex:
    jobs = [m for m in ep if position(m) in ("CENTER", "BASELINE")]
    for m, r in zip(jobs, ex.map(cycle_profile, jobs)):
        prof[(m["subject"], position(m))] = r
burst = (t_cyc >= 0) & (t_cyc < 70)
rows = []
for (s, pos), (rms, pw) in prof.items():
    rows.append({"subject": s, "position": pos, "LFP RMS 0-70 ms / rest (uV)": f"{rms[burst].mean():.2f} / {rms[~burst & (t_cyc >= 0)].mean():.2f}",
                 "LFP RMS peak lag (ms)": int(t_cyc[np.argmax(rms)]),
                 "1.4-1.5 kHz power 0-70 ms / rest": pw[burst].mean() / pw[~burst & (t_cyc >= 0)].mean(),
                 "1.4-1.5 kHz peak lag (ms)": int(t_cyc[np.argmax(pw)])})
AL = pd.DataFrame(rows).sort_values(["position", "subject"])
print(AL.round(3).to_string(index=False))
for pos in ("CENTER", "BASELINE"):
    a = AL[AL.position == pos]
    print(f"{pos}: 1.4-1.5 kHz power ratio 0-70 ms over the rest of the cycle, median {a['1.4-1.5 kHz power 0-70 ms / rest'].median():.3f}, "
          f"above 1.1 in {(a['1.4-1.5 kHz power 0-70 ms / rest'] > 1.1).sum()} of {len(a)} rats; "
          f"high-frequency peak within 0-70 ms in {((a['1.4-1.5 kHz peak lag (ms)'] >= 0) & (a['1.4-1.5 kHz peak lag (ms)'] < 70)).sum()}; "
          f"LFP RMS peak within 0-200 ms in {((a['LFP RMS peak lag (ms)'] >= 0) & (a['LFP RMS peak lag (ms)'] < 200)).sum()}")
mean_rms = {pos: np.mean([v[0] for k, v in prof.items() if k[1] == pos], axis=0) for pos in ("CENTER", "BASELINE")}
for pos, r in mean_rms.items():
    print(f"{pos}, mean over rats of the cycle RMS profile: 0-200 ms {r[(t_cyc >= 0) & (t_cyc < 200)].mean():.2f} uV, "
          f"200-1900 ms {r[t_cyc >= 200].mean():.2f} uV, max {r.max():.2f} uV at {int(t_cyc[np.argmax(r)])} ms")

In the PLV EEG the mean response after the interval start is many times the pre-onset level, so
the interval starts are the pulse onsets, in seconds (these files' tables are in seconds, unlike
the seizure labels). In the intracranial EP files the LFP shows no clear response at any lag of the
2 s cycle, in CENTER as in BASELINE (second table). The bursts can still be located: power at 1.4 to
1.5 kHz, the band of their 1.5 kHz pulse repetition frequency, peaks 10 to 60 ms after the interval
start in most CENTER recordings and in none of the resting ones (counts printed above), so the
interval starts are the burst onsets. An EP absent from the LFP is therefore not a misalignment of
the intervals.

## fig5b-tfus-evoked-potentials

> tFUS of the CC in wild-type rats evokes potentials in the left S1 LFP that differ significantly
> from the sham (resting) recording at some time points.

For each rat, the LFP around the first 200 bursts of the target-confirmation recording at the
chosen coordinates, and around pseudo onsets in the resting `BASELINE` recording, each trial
baseline-subtracted over -20 to 0 ms, averaged over trials. Readings of the claims file: the
channel, (a) the mean over the 64 channels or (b) each channel; the sham onsets, (a) every 2 s from
the start of the resting recording or (b) 200 onsets drawn at random (seed 0); the ANOVA, ordinary
or repeated measures (`tfus.timepoint_tests`; the repeated reading adds a random intercept per rat to
the same two-way ANOVA), Bonferroni over the time points from 0 to 200 ms.
The dandiset does not say which position was chosen. The paper says "the coordinates with the
clearest EP were used" (supplement, Target Confirmation); two readings are run: (a) the `CENTER`
recording, the first estimate, and (b) the position whose channel-mean EP has the largest peak
from 0 to 200 ms. Reading (b) picks the position on the data it then tests, so it is also run with
the position picked on the odd trials and the test made on the even trials (`largest, split`). The `BASELINE` file's own interval starts are shown as a third sham variant.

In [ ]:
rng = np.random.default_rng(0)
n0 = int(round(tfus.EP_PRE * tfus.RATE))
evo = {}          # (subject, position) -> evoked (time, channels)

evo_half = {}     # (subject, position) -> (evoked of the odd trials, evoked of the even trials)

def ep_job(m):
    on = np.sort(m["intervals"].start.to_numpy())[:tfus.N_TRIALS]
    x = tfus.read_epochs(m, on, threads=16)
    return (m["subject"], position(m)), tfus.evoked(x), (tfus.evoked(x[0::2]), tfus.evoked(x[1::2]))

t0 = time.time()
stim_files = [m for m in ep if position(m) not in ("BASELINE",) and not position(m).startswith("S1:")]
with ThreadPoolExecutor(4) as ex:
    for k, v, h in ex.map(ep_job, stim_files):
        evo[k], evo_half[k] = v, h
sham = {}
for m in [m for m in ep if position(m) == "BASELINE"]:
    T = m["n_samples"] / tfus.RATE
    grid = np.arange(2.0, T - tfus.EP_POST, 2.0)[:tfus.N_TRIALS]
    rand = np.sort(rng.uniform(tfus.EP_PRE, T - tfus.EP_POST, tfus.N_TRIALS))
    own = np.sort(m["intervals"].start.to_numpy())[:tfus.N_TRIALS]
    sham[m["subject"]] = {name: tfus.evoked(tfus.read_epochs(m, on, threads=16)) for name, on in
                          (("every 2 s", grid), ("random", rand), ("file intervals", own))}
    sham[m["subject"]]["n"] = {"every 2 s": len(grid), "random": len(rand), "file intervals": len(own)}
print(f"evoked potentials of {len(evo)} stimulated recordings and {len(sham)} resting recordings in {time.time() - t0:.0f} s")
print("Sham onsets per rat (every 2 s):", {s: v["n"]["every 2 s"] for s, v in sham.items()})

In [ ]:
t_ms = (np.arange(-n0, int(round(tfus.EP_POST * tfus.RATE))) / tfus.RATE) * 1000
post = t_ms >= 0
rats = sorted(sham)
peak = {k: np.abs(v.mean(axis=1)[post]).max() for k, v in evo.items()}
chosen_b = {s: max([k for k in evo if k[0] == s], key=lambda k: peak[k])[1] for s in rats}
peak_odd = {k: np.abs(v[0].mean(axis=1)[post]).max() for k, v in evo_half.items()}
chosen_split = {s: max([k for k in evo if k[0] == s], key=lambda k: peak_odd[k])[1] for s in rats}
print("Position with the largest channel-mean EP peak (reading b):", pd.Series(chosen_b).value_counts().to_dict())
print("Same, picked on the odd trials:", pd.Series(chosen_split).value_counts().to_dict(),
      f"; same position as reading (b) in {sum(chosen_b[s] == chosen_split[s] for s in rats)} of {len(rats)} rats")
rows, results = [], {}
for pos_reading in ("CENTER", "largest", "largest, split"):
    if pos_reading == "largest, split":
        A = np.stack([evo_half[(s, chosen_split[s])][1] for s in rats])    # even trials only
    else:
        keys = [(s, "CENTER" if pos_reading == "CENTER" else chosen_b[s]) for s in rats]
        A = np.stack([evo[k] for k in keys])                   # rats, time, channels
    for sham_reading in ("every 2 s", "random", "file intervals"):
        B = np.stack([sham[s][sham_reading] for s in rats])
        res = tfus.timepoint_tests(A.mean(axis=2)[:, post], B.mean(axis=2)[:, post])
        per_ch = [tfus.timepoint_tests(A[:, post, c], B[:, post, c]) for c in range(A.shape[2])]
        for anova in ("ordinary", "repeated"):
            col = f"p_{anova}"
            sig = res[col] < 0.05
            rows.append({"position": pos_reading, "sham onsets": sham_reading, "anova": anova,
                         "mean of 64: significant time points": int(sig.sum()), "of": len(res),
                         "first (ms)": round(float(t_ms[post][sig.to_numpy()][0]), 1) if sig.any() else None,
                         "smallest p": res[col].min(),
                         "channels with any significant point": sum(int((r[col] < 0.05).any()) for r in per_ch)})
        results[(pos_reading, sham_reading)] = (A, B, res)
TB = pd.DataFrame(rows)
print(TB.round(5).to_string(index=False))

In [ ]:
A, B, res = results[("CENTER", "every 2 s")]
a, b = A.mean(axis=2) * 1e6, B.mean(axis=2) * 1e6
fig, ax = plt.subplots(figsize=(7, 3))
for x, lab in ((a, "tFUS (CENTER)"), (b, "sham (every 2 s)")):
    m, s = x.mean(0), x.std(0) / np.sqrt(len(x))
    ax.plot(t_ms, m, label=lab); ax.fill_between(t_ms, m - s, m + s, alpha=0.3)
sig = np.zeros(len(t_ms), bool); sig[post] = res.p_repeated.to_numpy() < 0.05
ax.fill_between(t_ms, -100, 100, where=sig, color="gold", alpha=0.3, label="p < 0.05 (repeated)")
lim = np.abs(np.concatenate([a.mean(0), b.mean(0)])).max() * 1.5
ax.set_ylim(-lim, lim); ax.set_xlabel("ms from onset"); ax.set_ylabel("channel-mean LFP (uV)"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()
print(f"Plotted means: tFUS {a.mean(0).min():.2f} to {a.mean(0).max():.2f} uV, sham {b.mean(0).min():.2f} to {b.mean(0).max():.2f} uV; axis +/-{lim:.2f} uV holds both means")
print("Per rat, peak |channel-mean EP| from 0 to 200 ms (uV), CENTER against sham every 2 s:")
print(pd.DataFrame({"tFUS": np.abs(a[:, post]).max(1), "sham": np.abs(b[:, post]).max(1)}, index=rats).round(2).T.to_string())

**Result: blocked (methods_underspecified).** The readings the criterion names disagree. With the
mean over the 64 channels, two of the eighteen combinations of position, sham onsets and ANOVA find
significant time points, both with the CENTER position and random sham onsets (ordinary: 13 of 610
points from 66.5 ms, smallest Bonferroni p = 0.013; repeated measures: 26 points from 64.2 ms,
p = 0.00067); the other sixteen find none (smallest p = 0.054 to 1), including the largest-EP
position chosen on the odd trials and tested on the even ones. With each channel tested, 1 to 55
of the 64 channels have a significant point, depending on the combination. The mean channel-mean EP
at the CENTER position reaches 3.1 uV, the sham 1.7 uV. The intervals mark the burst onsets (the
1.4-1.5 kHz power peaks 10 to 60 ms after the interval start in 13 of 23 CENTER recordings and in
none of the resting ones), so the missing EP is not a misalignment. The criterion blocks the claim
when the readings disagree; which channels, which position and which sham onsets the paper used
are questions for its authors.

## Wild-type PLI

PLI (M6) of the PLV EEG files, first 200 pulses, in the windows pre (-20 to 0 ms), tFUS (0 to
20 ms) and post (20 to 200 ms), for every pair of one ipsilateral and one contralateral channel:
per trial and averaged (reading a) or over the concatenated samples of all trials (reading b).
The PLI of a short window is biased upwards, the more so the fewer samples it holds, so the 549
samples of 20 to 200 ms cannot be compared with the 61 of the pre window (the first version of this
attempt did, and its post-window decreases were that bias). The post window is therefore the mean
PLI of its nine 20 ms sub-windows, each as long as the pre window; the whole window is shown as
`post_full`. Each window's PLI is also given with a noise floor, the same PLI with the contralateral
channels taken from the next trial.
The change from pre is a difference (reading a) or a ratio (reading b), of the mean over the pairs
of a region (for the ratio, the region's mean PLI over its mean pre PLI, a standard choice, as
pair-wise ratios explode where the pre PLI is near 0). The dandiset's PLV files
are the recordings with 20 ms pulses, which fits the Methods' "tFUS (20 ms of stimulation)"; the
claims file reads the epochs as the 200 bursts of the target confirmation, whose EEG is not in the
dandiset. These files are used, and this is recorded as a deviation.

In [ ]:
t0 = time.time()
with ProcessPoolExecutor(3, mp_context=mp.get_context("spawn")) as ex:
    W = dict(zip([m["subject"] for m in plv], ex.map(tfus.wild_pli, plv)))
print(f"PLI of {len(W)} PLV files in {time.time() - t0:.0f} s; trials per file: {sorted({v['n_trials'] for v in W.values()})}")
k = len(tfus.S1_IPSI)
rows = []
for s, r in W.items():
    for band in tfus.BANDS:
        for reading in ("per_trial", "concat"):
            pre = r[band]["pre"][reading]
            for win in ("tfus", "post", "post_full"):
                x = r[band][win][reading]
                for norm in ("diff", "ratio"):
                    f = (lambda u, v: u - v) if norm == "diff" else (lambda u, v: u / v)
                    rows.append({"subject": s, "band": band, "reading": reading, "window": win, "norm": norm,
                                 "S1": f(x[:k, :k].mean(), pre[:k, :k].mean()), "nonS1": f(x[k:, k:].mean(), pre[k:, k:].mean()),
                                 "pairs decreasing": float((x < pre).mean()), "pre S1": pre[:k, :k].mean(), "pre nonS1": pre[k:, k:].mean()})
WP = pd.DataFrame(rows)
print("PLI before onset (pre window), mean over rats:")
print(WP[(WP.norm == "diff") & (WP.window == "tfus")].groupby(["band", "reading"])[["pre S1", "pre nonS1"]].mean().round(3).to_string())
print("\nMean over rats of the change from pre (S1 pairs, non-S1 pairs), and the fraction of all 256 pairs that decrease:")
WM = WP.groupby(["band", "reading", "norm", "window"])[["S1", "nonS1", "pairs decreasing"]].mean()
print(WM.round(4).unstack("window").to_string())
# Noise floor (verifier, round 1): the PLI of each window with the contralateral channels taken from the
# next trial, against the same-trial PLI. "post" is the mean over the nine 20 ms sub-windows of 20-200 ms,
# as long as the pre and tFUS windows; "post_full" is the whole 549-sample window.
rows = []
for s, r in W.items():
    for band in tfus.BANDS:
        for reading in ("per_trial", "concat"):
            for win in ("pre", "tfus", "post", "post_full"):
                x, f = r[band][win][reading], r[band][win][reading + "_shuffled"]
                rows.append({"subject": s, "band": band, "reading": reading, "window": win,
                             "S1": x[:k, :k].mean(), "S1 floor": f[:k, :k].mean(),
                             "nonS1": x[k:, k:].mean(), "nonS1 floor": f[k:, k:].mean()})
FL = pd.DataFrame(rows)
FLm = FL.groupby(["band", "reading", "window"])[["S1", "S1 floor", "nonS1", "nonS1 floor"]].mean()
print("\nPLI and its trial-shuffled noise floor, mean over rats:")
print(FLm.round(4).unstack("window").to_string())

## fig5cdj-wild-type-gamma-pli-decrease

> In wild-type rats, gamma band PLI decreases during and after tFUS relative to the pre-tFUS
> window in most brain regions, on average in both S1 and non-S1 regions, while it increases
> slightly in the sham.

The sham (the resting baseline recording of the target confirmation) has no EEG in the dandiset,
so the claim is blocked. The tFUS part is shown: the mean change of S1 and non-S1 pairs in both
windows, and the fraction of pairs that decrease.

In [ ]:
g = WM.loc["gamma"]
print(g.round(4).unstack("window").to_string())
print("\nGamma PLI against its trial-shuffled floor (means over rats):")
print(FLm.loc["gamma"].round(4).unstack("window").to_string())
per_rat = WP[(WP.band == "gamma") & (WP.norm == "diff")].pivot_table(index="subject", columns=["reading", "window"], values=["S1", "nonS1"])
per_rat = per_rat.loc[:, per_rat.columns.get_level_values("window") != "post_full"]
print("\nPer rat, gamma PLI difference from pre:")
print(per_rat.round(4).to_string())

**Result: blocked (data_missing).** Without the resting-state EEG the sham part cannot be tested.
The tFUS part does not hold either once the windows are of equal length. Per trial, the PLI equals
its trial-shuffled floor in every window (gamma S1 0.625 pre, 0.613 tFUS, 0.630 post, floors 0.628,
0.615, 0.629), so it measures the window length and not coupling. Over concatenated samples the
gamma PLI changes little: S1 +0.002 in the tFUS window and -0.008 after it, non-S1 +0.005 and
-0.002, with 47% of the 256 pairs decreasing in each window. The decreases the first version of
this attempt reported in the 20 to 200 ms window (S1 -0.029, non-S1 -0.016 concatenated, 100% of
pairs per trial) were the bias of comparing 549 samples with 61: they appear again with the whole
window (`post_full`) and vanish with its 20 ms sub-windows.

## fig5ghi-low-band-pli-s1-only

> In wild-type rats, theta, alpha and beta PLI decrease only within the S1 regions during tFUS.

"During the tFUS conditions" read as (a) the 0 to 20 ms window or (b) both windows. Reproduced
when, under one combination of readings, the mean normalized PLI of the S1 pairs is below the
pre-tFUS reference in all three bands and the non-S1 mean is not below it in any of them.

In [ ]:
rows = []
for reading in ("per_trial", "concat"):
    for norm in ("diff", "ratio"):
        ref = 0.0 if norm == "diff" else 1.0
        for win_reading, wins in (("a: 0-20 ms", ("tfus",)), ("b: both windows", ("tfus", "post"))):
            ok_s1, ok_ns = [], []
            for band in ("theta", "alpha", "beta"):
                for w in wins:
                    v = WM.loc[(band, reading, norm, w)]
                    ok_s1.append(v.S1 < ref); ok_ns.append(v.nonS1 >= ref)
            rows.append({"reading": reading, "norm": norm, "window": win_reading,
                         "S1 below in all": all(ok_s1), "S1 below (cells)": f"{sum(ok_s1)}/{len(ok_s1)}",
                         "non-S1 not below in all": all(ok_ns), "non-S1 not below (cells)": f"{sum(ok_ns)}/{len(ok_ns)}"})
print(pd.DataFrame(rows).to_string(index=False))
rows = []
for reading in ("per_trial", "concat"):
    for norm in ("diff", "ratio"):
        ref = 0.0 if norm == "diff" else 1.0
        for win_reading, wins in (("a: 0-20 ms", ("tfus",)), ("b: both windows", ("tfus", "post"))):
            bands_ok = [band for band in ("theta", "alpha", "beta")
                        if all(WM.loc[(band, reading, norm, w)].S1 < ref and WM.loc[(band, reading, norm, w)].nonS1 >= ref for w in wins)]
            rows.append({"reading": reading, "norm": norm, "window": win_reading, "bands with S1 below and non-S1 not below": ", ".join(bands_ok) or "none"})
print("\nPer band:")
print(pd.DataFrame(rows).to_string(index=False))
print("\nThe same with the whole 20-200 ms window (post_full), biased by its length (shown for comparison):")
rows = []
for reading in ("per_trial", "concat"):
    for norm in ("diff", "ratio"):
        ref = 0.0 if norm == "diff" else 1.0
        bands_ok = [band for band in ("theta", "alpha", "beta")
                    if all(WM.loc[(band, reading, norm, w)].S1 < ref and WM.loc[(band, reading, norm, w)].nonS1 >= ref for w in ("tfus", "post_full"))]
        rows.append({"reading": reading, "norm": norm, "bands with S1 below and non-S1 not below": ", ".join(bands_ok) or "none"})
print(pd.DataFrame(rows).to_string(index=False))
print("\nMeans over rats, theta, alpha, beta:")
print(WM.loc[["theta", "alpha", "beta"]].round(4).unstack("window").to_string())

**Result: not reproduced, resting on an assumption.** In no band and under no combination of readings does the S1 PLI fall
while the non-S1 PLI does not, with equal-length windows or with the whole 20 to 200 ms window. In
the 0 to 20 ms window the S1 PLI is not below pre in theta, alpha or beta under any reading
(differences +0.0002 to +0.070). After it, with 20 ms sub-windows over concatenated samples, both
regions fall slightly in alpha (S1 -0.010, non-S1 -0.014) and beta (-0.016, -0.026). The per-trial
PLI is at its trial-shuffled floor (theta S1 pre 0.976, floor 0.976), so the per-trial reading
cannot show a change, and the concatenated PLI (0.10 to 0.20) is about 0.02 above its floor. The outcome rests on the assumption that the PLV EEG files, the
recordings with 20 ms pulses, stand for the target-confirmation EEG that the claims file names and
the dandiset lacks; without it the claim would be blocked (data missing), as
fig5cdj-wild-type-gamma-pli-decrease is.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig5b-tfus-evoked-potentials | blocked: methods underspecified |
| fig5cdj-wild-type-gamma-pli-decrease | blocked: data missing |
| fig5ghi-low-band-pli-s1-only | not reproduced (assumed) |

The outcomes of the other nineteen claims are in `seizure_metrics.ipynb` and `eeg_blocks.ipynb`.

How the analysis differs from the paper's. The position whose bursts form the tFUS EPs is not
recorded; the CENTER recording and the position with the largest EP are both run. The wild-type
PLI uses the dandiset's PLV EEG files, recordings with 20 ms pulses, since the EEG of the
target-confirmation bursts is not in the dandiset; the Methods' "tFUS (20 ms of stimulation)" fits
these files; fig5ghi-low-band-pli-s1-only rests on this assumption. The repeated-measures reading of the EP test adds a random intercept per rat to the
group by time point ANOVA. Short windows bias the PLI upwards, so the 20 to 200 ms window is
compared with the pre window through its nine 20 ms sub-windows, and each PLI is given with a
trial-shuffled noise floor.

What was found about the dataset. The EP files of all 23 rats are there, one per transducer
position plus a resting recording, with interval tables in seconds whose starts are the burst
onsets (1.4-1.5 kHz power, the pulse repetition frequency, peaks just after them in most CENTER
recordings). The LFP shows no clear evoked response at any lag of the 2 s cycle. The resting recordings hold no EEG, so the sham of Figs. 5E-F cannot be computed.
The PLV EEG files cover 20 of the 23 rats (not BH638, BH642, BH644). The chosen position is not recorded.

In [ ]:
print(f"Notebook ran in {(time.time() - started) / 60:.1f} min")